In [1]:
import os
import csv
import numpy as np


# paths
THEFT_NPY_FOLDER  = r"N:\Final Preparation\FULL_DATASET\new\testing\positive\standardized_vedio\frame\lstm_dataset"
NORMAL_NPY_FOLDER = r"N:\Final Preparation\FULL_DATASET\new\testing\negitive\standardized_vedio\frame\lstm_dataset"
OUTPUT_CSV        = r"N:\Final Preparation\FULL_DATASET\new\testing\labels.csv"

EXPECTED_SHAPE = (30, 51)   # (WINDOW_SIZE, KEYPOINT_DIM)


class CSVLabelGenerator:

    def __init__(self, theft_folder, normal_folder, output_csv, expected_shape):
        self.theft_folder   = theft_folder
        self.normal_folder  = normal_folder
        self.output_csv     = output_csv
        self.expected_shape = expected_shape

        self.theft_files  = []
        self.normal_files = []

    def _scan_folder(self, folder, label_name):
        if not os.path.exists(folder):
            print(f"  folder not found: {folder}")
            return []

        all_files   = sorted([f for f in os.listdir(folder) if f.endswith(".npy")])
        valid       = []
        wrong_shape = []
        corrupt     = []

        for fname in all_files:
            fpath = os.path.join(folder, fname)
            try:
                data = np.load(fpath)
                if data.shape == self.expected_shape:
                    valid.append(fname)
                else:
                    wrong_shape.append((fname, data.shape))
            except Exception:
                corrupt.append(fname)

        print(f"[{label_name}] {len(valid)}/{len(all_files)} valid", end="")
        if wrong_shape:
            print(f"  |  {len(wrong_shape)} wrong shape (expected {self.expected_shape})", end="")
        if corrupt:
            print(f"  |  {len(corrupt)} corrupt", end="")
        print()

        if valid:
            sample   = np.load(os.path.join(folder, valid[0]))
            in_range = sample.min() >= 0.0 and sample.max() <= 1.0
            status   = "normalized" if in_range else "NOT normalized"
            print(f"  value range: [{sample.min():.3f}, {sample.max():.3f}]  ({status})")

        return valid

    def scan(self):
        self.theft_files  = self._scan_folder(self.theft_folder,  "THEFT")
        self.normal_files = self._scan_folder(self.normal_folder, "NORMAL")

    def save_csv(self):
        if not self.theft_files and not self.normal_files:
            print("No valid .npy files found — CSV not created")
            return

        rows = [{"filename": f, "label": 1} for f in self.theft_files]
        rows += [{"filename": f, "label": 0} for f in self.normal_files]

        os.makedirs(os.path.dirname(self.output_csv), exist_ok=True)
        with open(self.output_csv, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["filename", "label"])
            writer.writeheader()
            writer.writerows(rows)

        theft_count  = len(self.theft_files)
        normal_count = len(self.normal_files)
        ratio        = normal_count / max(theft_count, 1)

        print(f"\nCSV saved -> {self.output_csv}")
        print(f"  total: {len(rows)}  |  theft: {theft_count}  |  normal: {normal_count}  |  ratio: 1:{ratio:.1f}")

    def run(self):
        self.scan()
        self.save_csv()


generator = CSVLabelGenerator(
    theft_folder   = THEFT_NPY_FOLDER,
    normal_folder  = NORMAL_NPY_FOLDER,
    output_csv     = OUTPUT_CSV,
    expected_shape = EXPECTED_SHAPE,
)
generator.run()

[THEFT] 43/43 valid
  value range: [0.008, 1.000]  (normalized)
[NORMAL] 392/392 valid
  value range: [0.115, 1.000]  (normalized)

CSV saved -> N:\Final Preparation\FULL_DATASET\new\testing\labels.csv
  total: 435  |  theft: 43  |  normal: 392  |  ratio: 1:9.1
